# Project: mouse behaviour in an open field

Analyse animal trajectories: distance travelled, time in the center of the arena, and comparison of two groups.

The full explanations are in [open_field.md](open_field.md). *This work is licensed under [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0/).*

## How to use this notebook

* Run the cells **in order**, with `Shift+Enter`.
* Cells starting with `%%writefile` **create a file** (module, tests or script)
  in the folder of this notebook: like in the lesson, the project is made of
  real `.py` files, that you can also open in a text editor or Spyder.
* `%run script.py` runs a script as from the shell, and the cell with
  `pytest` runs the tests.
* Start JupyterLab from an **empty folder** dedicated to this project, in the
  `sci` environment:

  ```
  (sci) $ mkdir my_project
  (sci) $ cd my_project
  (sci) $ jupyter lab
  ```

## Step 1: the module

`simulate_trajectory` is a random walk in a square arena. `distance_travelled`
uses `np.diff` and `np.hypot`; `time_in_center` uses a boolean mask.

In [ ]:
%%writefile openfield.py
"""Analysis of mouse trajectories in an open-field test."""

import numpy as np


def simulate_trajectory(rng, duration=300, fps=10, arena=50, speed=8):
    """Simulate the trajectory of a mouse exploring a square arena.

    Parameters
    ----------
    rng : numpy.random.Generator
        Random generator.
    duration : float
        Duration of the recording (s).
    fps : int
        Number of positions per second.
    arena : float
        Side of the square arena (cm).
    speed : float
        Mean walking speed (cm/s).

    Returns
    -------
    t, x, y : numpy.ndarray
        Time (s) and position (cm) of the mouse.
    """
    n = int(duration * fps)
    t = np.arange(n) / fps
    x = np.empty(n)
    y = np.empty(n)
    x[0], y[0] = arena / 2, arena / 2
    angle = rng.uniform(0, 2 * np.pi)
    for i in range(1, n):
        angle += rng.normal(0, 0.4)                          # smooth turns
        step = rng.exponential(speed / fps)                  # variable speed
        x[i] = x[i - 1] + step * np.cos(angle)
        y[i] = y[i - 1] + step * np.sin(angle)
        if not (0 <= x[i] <= arena and 0 <= y[i] <= arena):  # hit a wall
            x[i] = np.clip(x[i], 0, arena)
            y[i] = np.clip(y[i], 0, arena)
            angle += np.pi / 2 + rng.uniform(0, np.pi)       # turn away
    return t, x, y


def distance_travelled(x, y):
    """Return the total length of the path (same unit as x and y)."""
    return np.sum(np.hypot(np.diff(x), np.diff(y)))


def time_in_center(t, x, y, arena=50, center=0.5):
    """Return the time spent in the central zone of the arena.

    The central zone is a square of side center * arena, in the middle.
    """
    margin = arena * (1 - center) / 2
    inside = ((x > margin) & (x < arena - margin) &
              (y > margin) & (y < arena - margin))
    dt = t[1] - t[0]
    return inside.sum() * dt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from openfield import simulate_trajectory, distance_travelled, time_in_center

t, x, y = simulate_trajectory(np.random.default_rng(0))
print("Distance:", distance_travelled(x, y), "cm")
print("Time in center:", time_in_center(t, x, y), "s")
plt.plot(x, y, linewidth=0.5)
plt.gca().set_aspect("equal")

## Step 2: tests

In [ ]:
%%writefile test_openfield.py
"""Tests for openfield.py."""

import numpy as np

from openfield import distance_travelled, simulate_trajectory, time_in_center


def test_distance_of_a_square():
    x = np.array([0, 10, 10, 0, 0])
    y = np.array([0, 0, 10, 10, 0])
    assert distance_travelled(x, y) == 40


def test_distance_diagonal():
    assert np.isclose(distance_travelled(np.array([0, 3]), np.array([0, 4])), 5)


def test_time_in_center():
    t = np.arange(4) * 0.5                  # 4 frames, 0.5 s apart
    x = np.array([25, 25, 2, 2])            # 2 frames in the center
    y = np.array([25, 25, 2, 2])
    assert time_in_center(t, x, y) == 1.0


def test_simulated_mouse_stays_in_arena():
    rng = np.random.default_rng(0)
    t, x, y = simulate_trajectory(rng, duration=60)
    assert x.min() >= 0 and x.max() <= 50
    assert y.min() >= 0 and y.max() <= 50

In [ ]:
import sys
!{sys.executable} -m pytest -q

## Step 3: an experiment

6 control mice and 6 treated mice, less active.

In [ ]:
%%writefile generate_mice.py
"""Simulate open-field recordings for two groups of mice."""

from pathlib import Path

import numpy as np

from openfield import simulate_trajectory

rng = np.random.default_rng(seed=11)
folder = Path("tracking")
folder.mkdir(exist_ok=True)
for group, speed in [("control", 8), ("treated", 5)]:
    for mouse in range(1, 7):
        t, x, y = simulate_trajectory(rng, speed=speed)
        filename = folder / f"{group}_mouse{mouse}.csv"
        np.savetxt(filename, np.column_stack([t, x, y]), delimiter=",",
                   fmt="%.2f", header="time,x,y", comments="")
print("Written 12 files in", folder)

In [ ]:
%run generate_mice.py

## Step 4: analysing all the recordings

In [ ]:
%%writefile analyze_mice.py
"""Analyse all the open-field recordings of a folder."""

import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from openfield import distance_travelled, time_in_center


def main():
    parser = argparse.ArgumentParser(description="Open-field analysis.")
    parser.add_argument("folder", help="folder with time,x,y CSV files")
    args = parser.parse_args()

    results = {}                                  # group -> list of distances
    with open("summary.csv", "w") as output:
        output.write("mouse,distance_cm,center_s\n")
        for filename in sorted(Path(args.folder).glob("*.csv")):
            t, x, y = np.loadtxt(filename, delimiter=",", skiprows=1, unpack=True)
            distance = distance_travelled(x, y)
            center = time_in_center(t, x, y)
            output.write(f"{filename.stem},{distance:.1f},{center:.1f}\n")
            print(f"{filename.stem:16s} distance {distance:7.1f} cm   "
                  f"center {center:5.1f} s")
            group = filename.stem.split("_")[0]
            if group not in results:
                results[group] = []
            results[group].append(distance)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    t, x, y = np.loadtxt(sorted(Path(args.folder).glob("*.csv"))[0],
                         delimiter=",", skiprows=1, unpack=True)
    axes[0].plot(x, y, linewidth=0.5)
    axes[0].set_aspect("equal")
    axes[0].set_title("Trajectory of the first mouse")
    axes[1].boxplot(list(results.values()))
    axes[1].set_xticks(range(1, len(results) + 1), list(results))
    axes[1].set_ylabel("Distance travelled (cm)")
    fig.tight_layout()
    fig.savefig("openfield.png", dpi=150)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
%run analyze_mice.py tracking

## Going further

**1.** Compute the instantaneous speed of a mouse (cm/s) and plot it as a function of time.

<details><summary>Answer</summary>

```python
t, x, y = np.loadtxt("tracking/control_mouse1.csv", delimiter=",",
                     skiprows=1, unpack=True)
speed = np.hypot(np.diff(x), np.diff(y)) / np.diff(t)
print("Mean speed:", speed.mean(), "cm/s")
plt.plot(t[1:], speed)
```

</details>

In [ ]:
# Your code here

**2.** Plot an occupancy heatmap of the arena with `plt.hist2d`.

<details><summary>Answer</summary>

```python
plt.hist2d(x, y, bins=20, range=[[0, 50], [0, 50]])
plt.colorbar(label="frames")
plt.gca().set_aspect("equal")
```

</details>

In [ ]:
# Your code here

**3.** Write `immobility_time(t, x, y, threshold=1.0)`, the time during which the speed is below a threshold.

<details><summary>Answer</summary>

```python
def immobility_time(t, x, y, threshold=1.0):
    speed = np.hypot(np.diff(x), np.diff(y)) / np.diff(t)
    return (speed < threshold).sum() * (t[1] - t[0])
```

</details>

In [ ]:
# Your code here